# 🗃️ Text-to-SQL as a Deterministic Workflow

## Learning Objectives
In this notebook, you will learn:
1. **The comic-store database** - Build a SQLite database from a script and read its schema the way the LLM will
2. **SQL generation with a prompt** - `create_sql_query_chain` + a SQLite prompt that returns only the query
3. **A fixed Text2SQL chain** - Generate SQL → execute it → turn the rows into an answer, with LCEL
4. **Measuring it** - Score the chain on questions whose correct answer we compute ourselves
5. **When the data disagrees** - The same question has two "right" answers in this database; a business rule in the prompt decides which one the chain uses

## Workflow Architecture
This is a **deterministic workflow**, not an autonomous agent: every question goes through the same three steps.

```
User Question → [Query Write Chain] → SQL Query → [Execute] → Rows → [Answer Chain] → Response
```

![Text2SQL Workflow](https://i.imgur.com/7WrLz9I.png)

## Where this fits
Prompt chaining (Module 05) is the pattern; this notebook applies it to a real task: answering business questions from a database. `05_text2sql_react_agent.ipynb` then solves the same task with an agent that explores the database itself, and compares the two.

## Prerequisites
- SQL basics; LCEL chains from `05_prompt_chaining/03_prompt_chaining_lcel.ipynb` (reading only)
- Dependencies installed with `uv sync` at the repository root (SQLite ships with Python — no `sqlite3` command-line tool needed)
- `OPENAI_API_KEY` in the project-root `.env`
- `comicdb_create_script.sql` next to this notebook (it is in the repository)

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: SQL generation and answer writing
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
---
## 🗄️ Part 1: The Comic Store Database

We'll use a **Comic Store Database** with the following tables:
- **Branch**: Store locations
- **Publisher**: Comic publishers
- **Comic**: Product catalog
- **Customer**: Customer information
- **Employee**: Staff data
- **Inventory**: Stock levels
- **Sale**: Transaction records
- **SaleTransactions**: Individual items in sales

![Database Schema](https://i.imgur.com/YzNCLpV.png)

In [ ]:
# ============================================================================
# DATABASE SETUP: build ComicStore.db from the SQL script next to this notebook
# ============================================================================
import sqlite3
from pathlib import Path

DB_PATH = Path("ComicStore.db")
DB_PATH.unlink(missing_ok=True)  # start from a fresh copy on every run

with sqlite3.connect(DB_PATH) as con:
    con.executescript(Path("comicdb_create_script.sql").read_text())
    tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]

print(f"✅ {DB_PATH} created with {len(tables)} tables: {tables}")

In [ ]:
# ============================================================================
# LANGCHAIN SQL DATABASE CONNECTION
# ============================================================================
# SQLDatabase is LangChain's wrapper for database interaction:
#   - db.run(sql): run a query          - db.get_table_info(): schema + sample rows
#   - db.get_usable_table_names(): list the tables
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

print(f"🔗 Database Dialect: {db.dialect}")
print(f"📋 Available Tables: {db.get_usable_table_names()}")
print("\n📚 Sample rows from Comic:")
print(db.run("SELECT * FROM Comic LIMIT 5;", include_columns=True))

In [ ]:
# ============================================================================
# UNDERSTANDING TABLE SCHEMA
# ============================================================================
# get_table_info() returns the CREATE TABLE statement and 3 sample rows per table.
# This is exactly what the LLM will see: column names and types, foreign keys, data format.
print(db.get_table_info(table_names=["Comic", "Sale"]))

---
## 🔧 Part 2: The Text2SQL Components

Two components turn a question into rows:
1. **SQL generation prompt + query write chain** — `create_sql_query_chain` fills the prompt's `{table_info}` with the schema and `{top_k}` with a default row limit, then asks the LLM for the query
2. **Query execution** — `QuerySQLDatabaseTool` runs a query and returns the rows as text

The prompt is the most important part: it must make the model return **only** the SQL. A reply like `SQLQuery: SELECT ...` or a fenced code block fails when executed.

In [ ]:
# ============================================================================
# COMPONENT 1: SQL GENERATION PROMPT
# ============================================================================
# The prompt is the MOST CRITICAL component of a Text2SQL system!
# 
# IMPORTANT: The prompt MUST tell the LLM to output ONLY the SQL query!
# If the LLM outputs "SQLQuery: SELECT..." instead of just "SELECT...",
# the QuerySQLDatabaseTool will fail with a syntax error.
#
# Key Prompt Engineering Techniques:
#   - Role assignment: "You are a SQLite expert"
#   - Specific instructions: "Never query for all columns"
#   - Output format: "Return ONLY the SQL query, nothing else"
#   - Guardrails: "Do not include sql markers or explanations"
# ============================================================================

from langchain_core.prompts.prompt import PromptTemplate

# The suffix contains placeholders that LangChain will populate automatically
# - {table_info}: Schema information for relevant tables
# - {input}: The user's natural language question
PROMPT_SUFFIX = """Only use the following tables:
{table_info}

Question: {input}"""

# Main prompt template for SQLite query generation
# CRITICAL: The prompt must instruct to return ONLY the SQL query!
_sqlite_prompt = """You are a SQLite expert.
Given an input question, create a syntactically correct SQLite query to run.

Unless the user specifies in the question a specific number of examples to obtain,
query for at most {top_k} results using the LIMIT clause as per SQLite.

You can order the results to return the most informative data in the database.
Never query for all columns from a table.

You must query only the columns that are needed to answer the question.
Wrap each column name in double quotes (") to denote them as delimited identifiers.

Pay attention to use only the column names you can see in the tables below.
Be careful to not query for columns that do not exist.
Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".
Pay attention to use table JOINS as necessary if you are adding relevant fields from different tables.

Return ONLY the SQL query as plain text, nothing else.
Do not include any prefixes like 'SQLQuery:' or 'SQL:'.
Do not include markdown code blocks or backticks.
Do not include any explanations.
Just output the raw SQL query starting with SELECT.

"""

# Combine the main prompt with suffix
SQLITE_PROMPT = PromptTemplate.from_template(_sqlite_prompt + PROMPT_SUFFIX)

print("✅ SQL Generation Prompt created!")
print(f"📋 Input variables required: {SQLITE_PROMPT.input_variables}")

In [ ]:
# ============================================================================
# COMPONENT 2: SQL QUERY GENERATION CHAIN
# ============================================================================
# create_sql_query_chain(): schema → prompt → LLM → SQL string
from langchain_classic.chains import create_sql_query_chain

query_write_chain = create_sql_query_chain(llm=llm, db=db, prompt=SQLITE_PROMPT, k=10)

generated_sql = query_write_chain.invoke({"question": "Top 5 most popular comics"})
print(f"Generated SQL:\n{generated_sql}")

In [ ]:
# ============================================================================
# COMPONENT 3: QUERY EXECUTION
# ============================================================================
from langchain_community.tools import QuerySQLDatabaseTool

execute_query_tool = QuerySQLDatabaseTool(db=db)

print(f"Rows:\n{execute_query_tool.invoke(generated_sql)}")

---
## 🔗 Part 3: The Complete Text2SQL Workflow

`RunnablePassthrough.assign()` adds one key to the running dictionary at each step: first `query`, then `result` (the rows of **that** query), then the `answer`. Keeping every intermediate value makes each run inspectable.

> **Key Insight**: execute the query you stored — `itemgetter("query") | execute_query_tool`. Piping the write chain into the executor a second time would generate a *second* query, and the SQL you display would not be the SQL whose rows produced the answer.

In [ ]:
# ============================================================================
# BUILD THE COMPLETE TEXT2SQL WORKFLOW
# ============================================================================
#   question ──► query (LLM) ──► result (run THAT query) ──► answer (LLM)
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result,
    answer the user question in a few sentences, using only the SQL result.

    Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    Answer:"""
)

text2sql_chain = (
    RunnablePassthrough.assign(query=query_write_chain)
    | RunnablePassthrough.assign(result=itemgetter("query") | execute_query_tool)
    | RunnablePassthrough.assign(answer=answer_prompt | llm | StrOutputParser())
)


def ask_database(question: str) -> dict:
    out = text2sql_chain.invoke({"question": question})
    print(f"📝 {question}\n🧾 SQL: {' '.join(out['query'].split())}\n💬 {out['answer']}\n")
    return out


_ = ask_database("Total number of customers")
_ = ask_database("What is the average price of a DC Comics comic?")

---
## 📏 Part 4: Measuring the Workflow

A demo that "looks right" is not a measurement. The evaluation set below pairs each question with SQL **we** wrote; running that SQL gives the expected answer. The chain passes a question when its answer contains the expected value.

In [ ]:
# ============================================================================
# EVALUATION SET: questions whose answer we compute ourselves with known SQL
# ============================================================================
import re

EVAL_SET = [
    ("How many customers do we have?",
     "SELECT COUNT(*) FROM Customer"),
    ("Which comic has sold the most copies?",
     """SELECT c.Title FROM SaleTransactions st JOIN Comic c ON c.ComicId = st.ComicId
        GROUP BY c.ComicId ORDER BY SUM(st.Quantity) DESC LIMIT 1"""),
    ("Which customer has spent the most money in total?",
     """SELECT cu.FirstName || ' ' || cu.LastName FROM Sale s
        JOIN Customer cu ON cu.CustomerId = s.CustomerId
        GROUP BY cu.CustomerId ORDER BY SUM(s.TotalAmount) DESC LIMIT 1"""),
    ("Which branch has the most comics in stock?",
     """SELECT b.Name FROM Inventory i JOIN Branch b ON b.BranchId = i.BranchId
        GROUP BY b.BranchId ORDER BY SUM(i.Stock) DESC LIMIT 1"""),
    ("Which publisher's comics generated the most sales revenue?",
     """SELECT p.Name FROM SaleTransactions st JOIN Comic c ON c.ComicId = st.ComicId
        JOIN Publisher p ON p.PublisherId = c.PublisherId
        GROUP BY p.PublisherId ORDER BY SUM(st.Quantity * st.Price) DESC LIMIT 1"""),
    ("Which customer has purchased the most comics?",
     """SELECT cu.FirstName || ' ' || cu.LastName FROM SaleTransactions st
        JOIN Sale s ON s.SaleId = st.SaleId JOIN Customer cu ON cu.CustomerId = s.CustomerId
        GROUP BY cu.CustomerId ORDER BY SUM(st.Quantity) DESC LIMIT 1"""),
    ("How many comics in the Superhero genre have never been sold?",
     """SELECT COUNT(*) FROM Comic WHERE Genre = 'Superhero'
        AND ComicId NOT IN (SELECT ComicId FROM SaleTransactions)"""),
]

with sqlite3.connect(DB_PATH) as con:
    EXPECTED = {q: str(con.execute(sql).fetchone()[0]) for q, sql in EVAL_SET}
    # Every name an answer could mention: to check that the expected one comes FIRST
    CANDIDATES = [r[0] for r in con.execute("""
        SELECT FirstName || ' ' || LastName FROM Customer UNION SELECT Title FROM Comic
        UNION SELECT Name FROM Branch UNION SELECT Name FROM Publisher""")]


def answer_is_correct(expected: str, answer: str) -> bool:
    """A number must appear as a whole number. A name must be the FIRST known name the
    answer mentions — listing six customers that include the right one is not an answer."""
    if expected.isdigit():
        return re.search(rf"\b{expected}\b", answer) is not None
    text = answer.lower()
    positions = {c: text.find(c.lower()) for c in CANDIDATES if c.lower() in text}
    return bool(positions) and min(positions, key=positions.get) == expected


for q, exp in EXPECTED.items():
    print(f"{q:<60} → {exp}")

In [ ]:
# ============================================================================
# SCORE THE WORKFLOW ON THE EVALUATION SET
# ============================================================================
passed = 0
for question, _ in EVAL_SET:
    out = text2sql_chain.invoke({"question": question})
    ok = answer_is_correct(EXPECTED[question], out["answer"])
    passed += ok
    print(f"{'✅' if ok else '❌'} {question}\n   expected: {EXPECTED[question]}\n   answer:   {out['answer'][:160]}\n")

print(f"Workflow score: {passed}/{len(EVAL_SET)}")

Read the ❌ lines, not just the score. The typical miss is **"Which customer has purchased the most comics?"**: the generated SQL counts line items (`COUNT(...)`) instead of summing `Quantity`, so a customer who bought several copies on one line is undercounted. The query runs, the answer reads well — and it is wrong. The counting question can go wrong a second way: the prompt's "at most 10 results" rule tempts the model to *list* titles with `LIMIT 10` and count the list. Only a ground-truth check catches either mistake.

---
## ⚖️ Part 5: When the Data Disagrees

Ask "who is the top salesperson?" and there are two defensible queries: sum `Sale.TotalAmount`, or sum the line items `SaleTransactions.Quantity × Price`. In a clean database they agree. In this one they do not — and the chain has no way to know which one the business means.

In [ ]:
# ============================================================================
# TWO DEFINITIONS OF "REVENUE" IN THE SAME DATABASE
# ============================================================================
with sqlite3.connect(DB_PATH) as con:
    mismatched = con.execute("""
        SELECT COUNT(*) FROM Sale s
        WHERE ABS(s.TotalAmount - (SELECT SUM(Quantity * Price) FROM SaleTransactions st
                                   WHERE st.SaleId = s.SaleId)) > 0.01""").fetchone()[0]
    by_header = con.execute("""
        SELECT e.FirstName || ' ' || e.LastName, ROUND(SUM(s.TotalAmount), 2) FROM Sale s
        JOIN Employee e ON e.EmployeeId = s.EmployeeId
        GROUP BY e.EmployeeId ORDER BY 2 DESC LIMIT 3""").fetchall()
    by_lines = con.execute("""
        SELECT e.FirstName || ' ' || e.LastName, ROUND(SUM(st.Quantity * st.Price), 2) FROM Sale s
        JOIN Employee e ON e.EmployeeId = s.EmployeeId
        JOIN SaleTransactions st ON st.SaleId = s.SaleId
        GROUP BY e.EmployeeId ORDER BY 2 DESC LIMIT 3""").fetchall()

print(f"Sales whose TotalAmount ≠ sum of their line items: {mismatched} of 20")
print(f"Top 3 by Sale.TotalAmount:        {by_header}")
print(f"Top 3 by SaleTransactions lines:  {by_lines}")

The top salesperson changes with the definition. Whatever the chain picks, it picks silently. The fix is not a smarter model but a **business rule** in the prompt: the team decides which source is authoritative and writes it down. Below, the same chain is rebuilt with one extra instruction, and the SQL it writes is checked in code.

In [ ]:
# ============================================================================
# ADD A BUSINESS RULE TO THE PROMPT
# ============================================================================
BUSINESS_RULES = """Business rules:
- Revenue is always computed from line items: SUM(SaleTransactions.Quantity * SaleTransactions.Price).
  Never use Sale.TotalAmount for revenue.

"""
ruled_prompt = PromptTemplate.from_template(_sqlite_prompt + BUSINESS_RULES + PROMPT_SUFFIX)
ruled_write_chain = create_sql_query_chain(llm=llm, db=db, prompt=ruled_prompt, k=10)

question = "Which are the top 3 salesmen with the highest revenue?"
for label, chain in [("original prompt", query_write_chain), ("with business rule", ruled_write_chain)]:
    sql = chain.invoke({"question": question})
    source = "line items" if "SaleTransactions" in sql else "Sale.TotalAmount"
    print(f"{label:<20} → revenue from {source}\n{' ' * 23}{' '.join(sql.split())}")

---
## 📝 Summary

In this notebook, we learned:

### 1. The database the model sees
- `SQLDatabase.get_table_info()` (CREATE statements + sample rows) is the model's only knowledge of the schema

### 2. SQL generation
- `create_sql_query_chain` + a prompt that demands raw SQL; format drift (`SQLQuery:` prefixes, code fences) is the most common failure

### 3. The fixed chain
- `RunnablePassthrough.assign` keeps question, SQL, rows and answer together; execute the stored query, never regenerate it

### 4. Measuring it
- Compute expected answers with your own SQL and score the chain against them

### 5. When the data disagrees
- Ambiguous business terms ("revenue") need explicit rules in the prompt; check the generated SQL in code

### Common Issues & Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| `near "SQLQuery": syntax error` | LLM outputs `SQLQuery: SELECT...` | Fix prompt to return ONLY the SQL |
| Empty results | Wrong table/column names | Check schema with `db.get_table_info()` |
| Wrong results | Incorrect JOINs | Provide clearer table relationships in prompt |

### Workflow vs Agent
This workflow offers a **deterministic, controllable** approach to Text2SQL:
- ✅ Fixed execution path (more predictable)
- ✅ Easier to debug and test
- ❌ Less flexible for complex queries
- ❌ No self-correction on errors

### Workflow vs Agent
This workflow offers a **deterministic, controllable** approach to Text2SQL:
- ✅ Fixed execution path (more predictable), easy to debug and test
- ✅ One SQL generation per question (cheap)
- ❌ No self-correction: a query that fails or returns nothing is not retried
- ❌ The whole schema goes into every prompt, however large the database

### Next Steps
- **`05_text2sql_react_agent.ipynb`**: the same task as a ReAct agent that explores the schema, checks its query and retries on errors — scored on the same evaluation set